# Pre-train AMRBART with MBart-50 (Vietnamese)

This notebook runs the 6-task AMR pre-training on Google Colab with a single GPU.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone repository

In [ ]:
!rm -rf /content/AMRBART
!git clone -b feat/mbart-50 https://github.com/Phucgiacat/AMRBART.git /content/AMRBART
!ls /content/AMRBART/pre-train/

## 3. Install dependencies

In [ ]:
# Install Rust (needed to build tokenizers 0.12.x from source for Python 3.12)
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
import os
os.environ['PATH'] = f"/root/.cargo/bin:{os.environ['PATH']}"
!rustc --version

!pip install -r /content/AMRBART/pre-train/requirements.txt

## 4. Download & prepare data

In [ ]:
!pip install -q gdown
!gdown --folder https://drive.google.com/drive/folders/10gwxpxAfha9zd1q6nakBBMohSBXGmmbC?usp=sharing -O /content/data

In [ ]:
!mkdir -p /content/AMRBART/pre-train/data/ViAMR
!cp /content/data/*.jsonl /content/AMRBART/pre-train/data/ViAMR/
!cp /content/AMRBART/pre-train/data/ViAMR/dev.jsonl /content/AMRBART/pre-train/data/ViAMR/val.jsonl
!ls -la /content/AMRBART/pre-train/data/ViAMR/

## 5. Download MBart-50 model

In [ ]:
!pip install -q huggingface_hub
!huggingface-cli download facebook/mbart-large-50 --local-dir /content/mbart-large-50

## 6. Verify setup

In [ ]:
import torch, transformers, datasets
print(f'torch:        {torch.__version__}')
print(f'transformers: {transformers.__version__}')
print(f'datasets:     {datasets.__version__}')
print(f'CUDA:         {torch.cuda.is_available()} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"})')

## 7. Run pre-training

In [ ]:
import sys, os

# Use sys.executable to get the actual python binary (not a wrapper script)
os.environ['PYTHON'] = sys.executable
os.environ['MODEL'] = '/content/mbart-large-50'

print(f"PYTHON={os.environ['PYTHON']}")

!cd /content/AMRBART/pre-train && bash run-posttrain-mbart50-vietnamese-6task-large-unified.sh